<a href="https://colab.research.google.com/github/hamza-24-ai/AI_Informed_Searches_Dynamic_PathFinding/blob/main/CA_Task_03_LSTM_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LSTM Train on CPU

In [ ]:
!pip -q install datasets

import torch, torch.nn as nn, torch.optim as optim, time, statistics, csv, resource
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

DEVICE = "cpu"            # change to "cuda" (T4) or "tpu" in each notebook
if DEVICE == "tpu":
    import torch_tpu

MAX_STEPS = 2000
BS, WIN, LR, SEED, WARM = 32, 20, 0.001, 0, 10
dev = torch.device(DEVICE)

ck = torch.load("lstm_wikitext2.pt", map_location="cpu")
vocab = ck["vocab"]; V = len(vocab)

class LSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedded = nn.Embedding(vocab_size, 100)
        self.lstm = nn.LSTM(100, 150, batch_first=True)
        self.linear = nn.Linear(150, vocab_size)
    def forward(self, x):
        emb = self.embedded(x)
        _, (final_hidden, _) = self.lstm(emb)
        return self.linear(final_hidden.squeeze(0))

# ---------- data ----------
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
def ids_of(split):
    toks = []
    for line in ds[split]["text"]:
        toks.extend(line.lower().split())
    return [vocab.get(w, 0) for w in toks]
def windows(ids):
    X = torch.tensor([ids[i:i+WIN] for i in range(len(ids)-WIN)])
    y = torch.tensor([ids[i+WIN] for i in range(len(ids)-WIN)])
    return X, y
Xtr, ytr = windows(ids_of("train"))
Xva, yva = windows(ids_of("validation"))
Xva, yva = Xva[:10000], yva[:10000]

class DS(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

gen = torch.Generator().manual_seed(SEED)
dl = DataLoader(DS(Xtr, ytr), batch_size=BS, shuffle=True, generator=gen)

# ---------- model: init on CPU, then move (same starting weights everywhere) ----------
torch.manual_seed(SEED)
model = LSTM(V)
model = model.to(dev)
opt = optim.Adam(model.parameters(), lr=LR)
crit = nn.CrossEntropyLoss()
print("device:", DEVICE, "| vocab:", V, "| params:", sum(p.numel() for p in model.parameters()))

# ---------- training (fixed steps) ----------
step_times, losses = [], []
step = 0
model.train()
t_total0 = time.perf_counter()
while step < MAX_STEPS:
    for xb, yb in dl:
        xb, yb = xb.to(dev), yb.to(dev)
        t0 = time.perf_counter()
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        opt.step()
        lv = loss.item()                       # forces device sync, so timing is complete
        step_times.append(time.perf_counter() - t0)
        losses.append(lv)
        step += 1
        if step % 500 == 0:
            print("step", step, "loss(avg last 100):", round(sum(losses[-100:]) / 100, 4))
        if step >= MAX_STEPS:
            break
total_time = time.perf_counter() - t_total0

# ---------- peak memory ----------
def peak_mem_mb():
    if DEVICE == "cuda":
        return torch.cuda.max_memory_allocated() / 1e6
    if DEVICE == "cpu":
        return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024   # whole process
    return None                                  # TPU: not measured with a verified API

# ---------- held-out accuracy ----------
model.eval(); correct = 0
with torch.inference_mode():
    for i in range(0, len(Xva), 256):
        out = model(Xva[i:i+256].to(dev)).float().cpu()
        correct += (out.argmax(1) == yva[i:i+256]).sum().item()
acc = correct / len(Xva)

steady = step_times[WARM:]
row = {"device": DEVICE, "steps": step, "total_time_s": total_time,
       "first_warmup_steps_s": sum(step_times[:WARM]),
       "step_median_s": statistics.median(steady),
       "step_mean_s": statistics.mean(steady),
       "step_std_s": statistics.stdev(steady),
       "peak_mem_MB": peak_mem_mb(),
       "final_loss_avg100": sum(losses[-100:]) / 100,
       "heldout_accuracy": acc}
print(row)
with open(f"lstm_train_{DEVICE}.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=row.keys())
    w.writeheader(); w.writerow(row)
print("Saved", f"lstm_train_{DEVICE}.csv")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/validation-00000-of-00(…): reconstructing file:   0%|          |  0.00B /  657kB            

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

device: cpu | vocab: 66650 | params: 16880350
step 500 loss(avg last 100): 7.6516
step 1000 loss(avg last 100): 7.3543
step 1500 loss(avg last 100): 7.1806
step 2000 loss(avg last 100): 7.0592
{'device': 'cpu', 'steps': 2000, 'total_time_s': 604.313570698, 'first_warmup_steps_s': 3.0766526579999436, 'step_median_s': 0.2771674845000689, 'step_mean_s': 0.3014967000633171, 'step_std_s': 0.047566529297474756, 'peak_mem_MB': 1435.68359375, 'final_loss_avg100': 7.059154720306396, 'heldout_accuracy': 0.1406}
Saved lstm_train_cpu.csv


# Train on GPU

In [ ]:
!pip -q install datasets

import torch, torch.nn as nn, torch.optim as optim, time, statistics, csv, resource
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

DEVICE = "cuda"            # change to "cuda" (T4) or "tpu" in each notebook
if DEVICE == "tpu":
    import torch_tpu

MAX_STEPS = 2000
BS, WIN, LR, SEED, WARM = 32, 20, 0.001, 0, 10
dev = torch.device(DEVICE)

ck = torch.load("lstm_wikitext2.pt", map_location="cpu")
vocab = ck["vocab"]; V = len(vocab)

class LSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedded = nn.Embedding(vocab_size, 100)
        self.lstm = nn.LSTM(100, 150, batch_first=True)
        self.linear = nn.Linear(150, vocab_size)
    def forward(self, x):
        emb = self.embedded(x)
        _, (final_hidden, _) = self.lstm(emb)
        return self.linear(final_hidden.squeeze(0))

# ---------- data ----------
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
def ids_of(split):
    toks = []
    for line in ds[split]["text"]:
        toks.extend(line.lower().split())
    return [vocab.get(w, 0) for w in toks]
def windows(ids):
    X = torch.tensor([ids[i:i+WIN] for i in range(len(ids)-WIN)])
    y = torch.tensor([ids[i+WIN] for i in range(len(ids)-WIN)])
    return X, y
Xtr, ytr = windows(ids_of("train"))
Xva, yva = windows(ids_of("validation"))
Xva, yva = Xva[:10000], yva[:10000]

class DS(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

gen = torch.Generator().manual_seed(SEED)
dl = DataLoader(DS(Xtr, ytr), batch_size=BS, shuffle=True, generator=gen)

# ---------- model: init on CPU, then move (same starting weights everywhere) ----------
torch.manual_seed(SEED)
model = LSTM(V)
model = model.to(dev)
opt = optim.Adam(model.parameters(), lr=LR)
crit = nn.CrossEntropyLoss()
print("device:", DEVICE, "| vocab:", V, "| params:", sum(p.numel() for p in model.parameters()))

# ---------- training (fixed steps) ----------
step_times, losses = [], []
step = 0
model.train()
t_total0 = time.perf_counter()
while step < MAX_STEPS:
    for xb, yb in dl:
        xb, yb = xb.to(dev), yb.to(dev)
        t0 = time.perf_counter()
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        opt.step()
        lv = loss.item()                       # forces device sync, so timing is complete
        step_times.append(time.perf_counter() - t0)
        losses.append(lv)
        step += 1
        if step % 500 == 0:
            print("step", step, "loss(avg last 100):", round(sum(losses[-100:]) / 100, 4))
        if step >= MAX_STEPS:
            break
total_time = time.perf_counter() - t_total0

# ---------- peak memory ----------
def peak_mem_mb():
    if DEVICE == "cuda":
        return torch.cuda.max_memory_allocated() / 1e6
    if DEVICE == "cpu":
        return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024   # whole process
    return None                                  # TPU: not measured with a verified API

# ---------- held-out accuracy ----------
model.eval(); correct = 0
with torch.inference_mode():
    for i in range(0, len(Xva), 256):
        out = model(Xva[i:i+256].to(dev)).float().cpu()
        correct += (out.argmax(1) == yva[i:i+256]).sum().item()
acc = correct / len(Xva)

steady = step_times[WARM:]
row = {"device": DEVICE, "steps": step, "total_time_s": total_time,
       "first_warmup_steps_s": sum(step_times[:WARM]),
       "step_median_s": statistics.median(steady),
       "step_mean_s": statistics.mean(steady),
       "step_std_s": statistics.stdev(steady),
       "peak_mem_MB": peak_mem_mb(),
       "final_loss_avg100": sum(losses[-100:]) / 100,
       "heldout_accuracy": acc}
print(row)
with open(f"lstm_train_{DEVICE}.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=row.keys())
    w.writeheader(); w.writerow(row)
print("Saved", f"lstm_train_{DEVICE}.csv")

# Train on TPU

In [1]:
!pip -q install datasets

import torch, torch.nn as nn, torch.optim as optim, time, statistics, csv, resource
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

DEVICE = "tpu"            # change to "cuda" (T4) or "tpu" in each notebook
if DEVICE == "tpu":
    import torch_tpu

MAX_STEPS = 2000
BS, WIN, LR, SEED, WARM = 32, 20, 0.001, 0, 10
dev = torch.device(DEVICE)

ck = torch.load("lstm_wikitext2.pt", map_location="cpu")
vocab = ck["vocab"]; V = len(vocab)

class LSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedded = nn.Embedding(vocab_size, 100)
        self.lstm = nn.LSTM(100, 150, batch_first=True)
        self.linear = nn.Linear(150, vocab_size)
    def forward(self, x):
        emb = self.embedded(x)
        _, (final_hidden, _) = self.lstm(emb)
        return self.linear(final_hidden.squeeze(0))

# ---------- data ----------
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
def ids_of(split):
    toks = []
    for line in ds[split]["text"]:
        toks.extend(line.lower().split())
    return [vocab.get(w, 0) for w in toks]
def windows(ids):
    X = torch.tensor([ids[i:i+WIN] for i in range(len(ids)-WIN)])
    y = torch.tensor([ids[i+WIN] for i in range(len(ids)-WIN)])
    return X, y
Xtr, ytr = windows(ids_of("train"))
Xva, yva = windows(ids_of("validation"))
Xva, yva = Xva[:10000], yva[:10000]

class DS(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

gen = torch.Generator().manual_seed(SEED)
dl = DataLoader(DS(Xtr, ytr), batch_size=BS, shuffle=True, generator=gen)

# ---------- model: init on CPU, then move (same starting weights everywhere) ----------
torch.manual_seed(SEED)
model = LSTM(V)
model = model.to(dev)
opt = optim.Adam(model.parameters(), lr=LR)
crit = nn.CrossEntropyLoss()
print("device:", DEVICE, "| vocab:", V, "| params:", sum(p.numel() for p in model.parameters()))

# ---------- training (fixed steps) ----------
step_times, losses = [], []
step = 0
model.train()
t_total0 = time.perf_counter()
while step < MAX_STEPS:
    for xb, yb in dl:
        xb, yb = xb.to(dev), yb.to(dev)
        t0 = time.perf_counter()
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        opt.step()
        lv = loss.item()                       # forces device sync, so timing is complete
        step_times.append(time.perf_counter() - t0)
        losses.append(lv)
        step += 1
        if step % 500 == 0:
            print("step", step, "loss(avg last 100):", round(sum(losses[-100:]) / 100, 4))
        if step >= MAX_STEPS:
            break
total_time = time.perf_counter() - t_total0

# ---------- peak memory ----------
def peak_mem_mb():
    if DEVICE == "cuda":
        return torch.cuda.max_memory_allocated() / 1e6
    if DEVICE == "cpu":
        return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024   # whole process
    return None                                  # TPU: not measured with a verified API

# ---------- held-out accuracy ----------
model.eval(); correct = 0
with torch.inference_mode():
    for i in range(0, len(Xva), 256):
        out = model(Xva[i:i+256].to(dev)).float().cpu()
        correct += (out.argmax(1) == yva[i:i+256]).sum().item()
acc = correct / len(Xva)

steady = step_times[WARM:]
row = {"device": DEVICE, "steps": step, "total_time_s": total_time,
       "first_warmup_steps_s": sum(step_times[:WARM]),
       "step_median_s": statistics.median(steady),
       "step_mean_s": statistics.mean(steady),
       "step_std_s": statistics.stdev(steady),
       "peak_mem_MB": peak_mem_mb(),
       "final_loss_avg100": sum(losses[-100:]) / 100,
       "heldout_accuracy": acc}
print(row)
with open(f"lstm_train_{DEVICE}.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=row.keys())
    w.writeheader(); w.writerow(row)
print("Saved", f"lstm_train_{DEVICE}.csv")

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

... (4 outputs left out)

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

device: tpu | vocab: 66650 | params: 16880350


/tmp/ipykernel_624/3101391389.py:73: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at external/rules_python++pip+torch_tpu_pypi_2_14_0_314_torch/site-packages/torch/include/ATen/core/TensorBody.h:494.)
  lv = loss.item()                       # forces device sync, so timing is complete


step 500 loss(avg last 100): 7.6503
step 1000 loss(avg last 100): 7.3556
step 1500 loss(avg last 100): 7.1796
step 2000 loss(avg last 100): 7.0613
{'device': 'tpu', 'steps': 2000, 'total_time_s': 7.481207804999997, 'first_warmup_steps_s': 1.573017170999833, 'step_median_s': 0.002215059499974359, 'step_mean_s': 0.0022076247427137833, 'step_std_s': 0.00010734905486226992, 'peak_mem_MB': None, 'final_loss_avg100': 7.061315679550171, 'heldout_accuracy': 0.1379}
Saved lstm_train_tpu.csv
